# Classifier-free guidance on Tahoe cell states

The base diffusion model for Modality 2 and its guidance demonstration. **The model, training
loop, guided sampler and visual checks are a teammate's `Tahoe_PCA_Diffusion_CFG.ipynb`, kept
as written.** What this notebook adds: the loader from `src/guidance/data/cells.py` (the same
`TahoePCs` class, plus the dose files of the current data zip), the visual checks moved from
training pairs to **held-out validation pairs**, an evaluation of the samples (MMD, energy
distance, variance ratio against real validation cells, per pair, per guidance strength), and
a manifest with the commit, config, hardware, training time and every number.

Conditioning is the (drug, cell line) pair as two embeddings summed with the time embedding,
so a pair never seen in training is still a seen drug plus a seen line. Classifier-free
guidance: `eps = eps_uncond + w (eps_cond - eps_uncond)`, with the null condition dropping
drug and line together during training.

Run top to bottom on a GPU runtime. Any setting below can be overridden from the environment,
e.g. `CFG_EPOCHS=5 CFG_LIMIT=20000` for a smoke run.

In [ ]:
# Bootstrap. Run once.
import importlib.util, subprocess, sys, os, time

for pkg in ("diffusers",):
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

if importlib.util.find_spec("guidance") is None:
    if not os.path.exists("discrete-generative-models-proj-1"):
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/imperorrp/discrete-generative-models-proj-1.git"], check=True)
    sys.path.insert(0, "discrete-generative-models-proj-1/src")

import json
import math
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from diffusers import DDPMScheduler
from tqdm.auto import tqdm

def _git_sha():
    for d in ("discrete-generative-models-proj-1", "."):
        try:
            return subprocess.check_output(["git", "-C", d, "rev-parse", "--short", "HEAD"],
                                           text=True, stderr=subprocess.DEVNULL).strip()
        except Exception:
            pass
    return "unknown"

GIT_SHA = _git_sha()
print("code version:", GIT_SHA, "| PyTorch:", torch.__version__)

## Settings

Theirs, unchanged, plus environment overrides for headless runs.

In [ ]:
def _env(name, default):
    v = os.environ.get(f"CFG_{name}")
    return default if v is None else type(default)(v)

BATCH_SIZE = _env("BATCH_SIZE", 256)
SEED = _env("SEED", 42)
NUM_TRAIN_TIMESTEPS = 1000
CFG_DROPOUT_PROB = 0.1     # probability of training on the null (unconditional) id pair, as in cifar_diffusion.py
EPOCHS = _env("EPOCHS", 50)
LR = 2e-4
LIMIT = _env("LIMIT", 0)                    # >0 truncates both splits, for smoke runs
SAMPLE_STEPS = _env("SAMPLE_STEPS", 250)    # their sampler's default
GUIDANCE_SWEEP = [float(v) for v in os.environ.get("CFG_GUIDANCE_SWEEP", "0,1,3,6").split(",")]
N_EVAL_PAIRS = _env("N_EVAL_PAIRS", 6)
REUSE_CKPT = _env("REUSE_CKPT", 1)          # load best.pt from OUT_DIR if present instead of retraining
OUT_DIR = Path("checkpoints/diffusion_cfg")

torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Device:", DEVICE, "| epochs:", EPOCHS, "| batch:", BATCH_SIZE, "| sweep:", GUIDANCE_SWEEP)

## Data

Their `TahoePCs` loader, imported from the package (identical code; it also reads the dose
files of the current zip). The split holds out about 20% of (cell line, drug) pairs; every
validation cell is a combination never seen in training. Upload `25lines_100drugs_pca_data_v2.zip`
and `!unzip -q` it, or mount Drive, or set `TAHOE_PCA_DIR`.

In [ ]:
from torch.utils.data import BatchSampler, DataLoader, RandomSampler, SequentialSampler
from guidance.data.cells import TahoePCs, load_pca_model, pcs_to_hvg_expression, find_pca_dir, condition_texts

PCA_DIR = find_pca_dir()
print("PCA_DIR:", PCA_DIR)

def truncate(ds, n):
    '''Keep the first n cells of a TahoePCs dataset, for smoke runs only.'''
    for name in ("pcs", "cluster", "drug_id", "line_id", "dose", "is_control"):
        setattr(ds, name, getattr(ds, name)[:n])

# Their make_pca_loaders, with the optional truncation applied before the samplers are built.
train_ds = TahoePCs("train", PCA_DIR)
val_ds = TahoePCs("validation", PCA_DIR, pc_scale=train_ds.pc_scale)
if LIMIT:
    truncate(train_ds, LIMIT); truncate(val_ds, LIMIT)
generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=None, num_workers=0,
                          sampler=BatchSampler(RandomSampler(train_ds, generator=generator), BATCH_SIZE, drop_last=True))
validation_loader = DataLoader(val_ds, batch_size=None, num_workers=0,
                               sampler=BatchSampler(SequentialSampler(val_ds), BATCH_SIZE, drop_last=False))
print("Training cells:", len(train_loader.dataset), "| Validation cells:", len(validation_loader.dataset))

DATA_DIM = train_loader.dataset.pcs.shape[1]
drug_names = train_loader.dataset.drug_names.tolist()
line_names = train_loader.dataset.line_names.tolist()
drug_to_id = {name: i for i, name in enumerate(drug_names)}
line_to_id = {name: i for i, name in enumerate(line_names)}
NULL_DRUG_ID, NULL_LINE_ID = len(drug_names), len(line_names)
DRUG_TEXT, LINE_TEXT = condition_texts(PCA_DIR)        # for readable tables, from the Tahoe metadata

batch = next(iter(train_loader))
print("pcs:", tuple(batch["pcs"].shape), "| e.g. drug:", batch["drug"][0], "| e.g. cell line:", batch["cell_line_id"][0])
print(f"{len(drug_names)} drugs (null id {NULL_DRUG_ID}), {len(line_names)} cell lines (null id {NULL_LINE_ID}), "
      f"data dim {DATA_DIM}")

## Model

Theirs, verbatim.

In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        half = dim // 2
        freqs = torch.exp(-math.log(10000.0) * torch.arange(half) / max(half - 1, 1))
        self.register_buffer("freqs", freqs, persistent=False)

    def forward(self, t):
        args = t.float()[:, None] * self.freqs[None, :].to(t.device)
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)


class ResidualBlock(nn.Module):
    def __init__(self, dim, cond_dim):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.fc1 = nn.Linear(dim + cond_dim, dim)
        self.fc2 = nn.Linear(dim, dim)
        self.act = nn.SiLU()

    def forward(self, x, cond):
        h = torch.cat([self.norm(x), cond], dim=-1)
        h = self.fc2(self.act(self.fc1(h)))
        return x + h


class DiffusionMLP(nn.Module):
    def __init__(self, data_dim, n_drugs, n_lines, time_dim=128, cond_dim=128, hidden_dim=256, n_blocks=4):
        super().__init__()
        self.data_dim = data_dim
        self.time_embed = SinusoidalTimeEmbedding(time_dim)
        self.time_mlp = nn.Sequential(nn.Linear(time_dim, cond_dim), nn.SiLU(), nn.Linear(cond_dim, cond_dim))
        self.drug_embed = nn.Embedding(n_drugs + 1, cond_dim)   # last row is the null/unconditional id
        self.line_embed = nn.Embedding(n_lines + 1, cond_dim)   # last row is the null/unconditional id
        self.input_proj = nn.Linear(data_dim, hidden_dim)
        self.blocks = nn.ModuleList([ResidualBlock(hidden_dim, cond_dim) for _ in range(n_blocks)])
        self.output_proj = nn.Linear(hidden_dim, data_dim)

    def forward(self, x, t, drug_id, line_id):
        cond = self.time_mlp(self.time_embed(t)) + self.drug_embed(drug_id) + self.line_embed(line_id)
        h = self.input_proj(x)
        for block in self.blocks:
            h = block(h, cond)
        return self.output_proj(h)

## Scheduler, model, optimizer

Theirs: `DDPMScheduler`, cosine beta schedule, epsilon prediction, no sample clipping.

In [ ]:
scheduler = DDPMScheduler(num_train_timesteps=NUM_TRAIN_TIMESTEPS, beta_schedule="squaredcos_cap_v2",
                          prediction_type="epsilon", clip_sample=False)
model = DiffusionMLP(DATA_DIM, n_drugs=len(drug_names), n_lines=len(line_names)).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
N_PARAMS = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {N_PARAMS:,}")

## Training and validation steps

Theirs. One coin per cell drops *both* ids to their null id together (joint dropout). Validation
always uses the true condition and a fixed generator, so it is comparable across epochs.

In [ ]:
def encode_condition(batch, device):
    drug_id = torch.tensor([drug_to_id[d] for d in batch["drug"]], dtype=torch.long, device=device)
    line_id = torch.tensor([line_to_id[c] for c in batch["cell_line_id"]], dtype=torch.long, device=device)
    return drug_id, line_id


def train_one_epoch(model, loader, scheduler, optimizer, scaler, device, cfg_dropout_prob):
    model.train()
    use_amp = device.type == "cuda"
    total_loss, count = 0.0, 0
    for batch in tqdm(loader, desc="train", leave=False):
        x = batch["pcs"].to(device, non_blocking=True)
        drug_id, line_id = encode_condition(batch, device)
        drop = torch.rand(drug_id.shape, device=device) < cfg_dropout_prob
        drug_id = torch.where(drop, torch.full_like(drug_id, NULL_DRUG_ID), drug_id)
        line_id = torch.where(drop, torch.full_like(line_id, NULL_LINE_ID), line_id)
        noise = torch.randn_like(x)
        t = torch.randint(0, scheduler.config.num_train_timesteps, (x.shape[0],), device=device)
        noisy_x = scheduler.add_noise(x, noise, t)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, enabled=use_amp):
            predicted_noise = model(noisy_x, t, drug_id, line_id)
            loss = F.mse_loss(predicted_noise.float(), noise.float())
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * x.shape[0]
        count += x.shape[0]
    return total_loss / count


@torch.inference_mode()
def evaluate(model, loader, scheduler, device, seed=0):
    model.eval()
    generator = torch.Generator(device=device).manual_seed(seed)
    total_loss, count = 0.0, 0
    for batch in loader:
        x = batch["pcs"].to(device)
        drug_id, line_id = encode_condition(batch, device)
        noise = torch.randn(x.shape, device=device, generator=generator)
        t = torch.randint(0, scheduler.config.num_train_timesteps, (x.shape[0],), device=device, generator=generator)
        noisy_x = scheduler.add_noise(x, noise, t)
        predicted_noise = model(noisy_x, t, drug_id, line_id)
        loss = F.mse_loss(predicted_noise, noise)
        total_loss += loss.item() * x.shape[0]
        count += x.shape[0]
    return total_loss / count

## Run training

Theirs, plus: time per epoch, the commit and the split recorded in the checkpoint, and a
reload of `best.pt` if it is already in `checkpoints/diffusion_cfg/` (upload it to skip the
50 epochs). Whichever path ran, sampling below uses the best-validation weights.

In [ ]:
CKPT_META = dict(split="pair_seed42_val20_raw4", n_drugs=len(drug_names), n_lines=len(line_names), data_dim=DATA_DIM,
                 num_train_timesteps=NUM_TRAIN_TIMESTEPS, cfg_dropout_prob=CFG_DROPOUT_PROB, n_params=N_PARAMS)
history, TRAIN_SECONDS = [], 0.0

if REUSE_CKPT and (OUT_DIR / "best.pt").exists():
    ck = torch.load(OUT_DIR / "best.pt", map_location=DEVICE, weights_only=False)
    mismatch = {k: (ck.get("meta", {}).get(k), v) for k, v in CKPT_META.items() if ck.get("meta", {}).get(k) != v}
    assert not mismatch, f"best.pt was trained under a different setup: {mismatch}"
    model.load_state_dict(ck["model"]); best_val = ck["val_loss"]; best_epoch = ck["epoch"]
    history = ck.get("history", [])
    print(f"loaded best.pt (epoch {best_epoch}, val {best_val:.4f}); skipping training")
else:
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
    best_val, best_epoch = float("inf"), 0
    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        train_loss = train_one_epoch(model, train_loader, scheduler, optimizer, scaler, DEVICE, CFG_DROPOUT_PROB)
        val_loss = evaluate(model, validation_loader, scheduler, DEVICE)
        seconds = time.time() - t0; TRAIN_SECONDS += seconds
        history.append(dict(epoch=epoch, train_loss=train_loss, val_loss=val_loss, seconds=seconds))
        print(f"epoch {epoch:>3}/{EPOCHS}  train mse {train_loss:.4f}  val mse {val_loss:.4f}  ({seconds:.0f}s)")
        checkpoint = dict(model=model.state_dict(), optimizer=optimizer.state_dict(), epoch=epoch, val_loss=val_loss,
                          drug_names=drug_names, line_names=line_names, data_dim=DATA_DIM, meta=CKPT_META,
                          code_version=GIT_SHA, history=history,
                          config=dict(num_train_timesteps=NUM_TRAIN_TIMESTEPS, cfg_dropout_prob=CFG_DROPOUT_PROB))
        torch.save(checkpoint, OUT_DIR / "latest.pt")
        if val_loss < best_val:
            best_val, best_epoch = val_loss, epoch
            torch.save(checkpoint, OUT_DIR / "best.pt")
    print("Best validation MSE:", best_val, "at epoch", best_epoch)
    model.load_state_dict(torch.load(OUT_DIR / "best.pt", map_location=DEVICE, weights_only=False)["model"])
model.eval()

## Sampling with classifier-free guidance

Theirs, verbatim. `eps = eps_uncond + guidance_scale * (eps_cond - eps_uncond)`. `guidance_scale = 1`
is plain conditional sampling; `0` is fully unconditional; above 1 pushes samples toward the
condition at some cost to diversity. Ancestral DDPM over `SAMPLE_STEPS` steps: two network
evaluations per step whenever the scale is not 1, so NFE is `2 * SAMPLE_STEPS`. The other cell
notebooks sample with 50 deterministic DDIM steps; that difference is disclosed wherever numbers
from this notebook sit next to theirs.

In [ ]:
@torch.inference_mode()
def sample(model, scheduler, drug_name, line_name, num_samples, guidance_scale, data_dim, device,
          steps=250, seed=0):
    model.eval()
    generator = torch.Generator(device=device).manual_seed(seed)
    local_scheduler = DDPMScheduler.from_config(scheduler.config)
    local_scheduler.set_timesteps(steps, device=device)
    x = torch.randn((num_samples, data_dim), device=device, generator=generator)
    drug_id = torch.full((num_samples,), drug_to_id[drug_name], dtype=torch.long, device=device)
    line_id = torch.full((num_samples,), line_to_id[line_name], dtype=torch.long, device=device)
    null_drug = torch.full_like(drug_id, NULL_DRUG_ID)
    null_line = torch.full_like(line_id, NULL_LINE_ID)
    for t in local_scheduler.timesteps:
        t_batch = torch.full((num_samples,), int(t), dtype=torch.long, device=device)
        if guidance_scale == 1:
            eps = model(x, t_batch, drug_id, line_id)
        else:
            eps_uncond = model(x, t_batch, null_drug, null_line)
            eps_cond = model(x, t_batch, drug_id, line_id)
            eps = eps_uncond + guidance_scale * (eps_cond - eps_uncond)
        x = local_scheduler.step(eps, t, x, generator=generator).prev_sample
    return x

## Which pairs to look at: held-out validation pairs

Their checks drew pairs from the training set. Here every pair is a **validation** pair: a
(cell line, drug) combination the model never saw, so what follows measures generalisation.
The treated validation pairs with the most cells are used, so each has enough real cells for a
distribution comparison.

In [ ]:
def eligible_pairs(dataset, min_cells=30, dmso_id=0):
    '''(line_id, drug_id) treated pairs with at least min_cells real cells, most cells first.'''
    line_id, drug_id = torch.as_tensor(dataset.line_id), torch.as_tensor(dataset.drug_id)
    treated = drug_id != dmso_id
    pairs, counts = torch.unique(torch.stack([line_id[treated], drug_id[treated]], dim=1), dim=0, return_counts=True)
    order = counts.argsort(descending=True)
    return pairs[order][counts[order] >= min_cells], counts[order][counts[order] >= min_cells]

val_ds = validation_loader.dataset
VAL_PAIRS, VAL_COUNTS = eligible_pairs(val_ds)
EVAL_PAIRS = VAL_PAIRS[:N_EVAL_PAIRS].tolist()
print(f"{len(VAL_PAIRS)} treated validation pairs with >= 30 cells; evaluating on the {len(EVAL_PAIRS)} largest:")
for (l, d), n in zip(EVAL_PAIRS, VAL_COUNTS[:N_EVAL_PAIRS].tolist()):
    print(f"   {LINE_TEXT[line_names[l]]:28s} + {DRUG_TEXT[drug_names[d]][:50]:50s} n={n}")

def real_cells(dataset, line_id, drug_id):
    mask = (dataset.line_id == line_id) & (dataset.drug_id == drug_id)
    return dataset.pcs[torch.from_numpy(mask)]

DEMO_LINE_ID, DEMO_DRUG_ID = EVAL_PAIRS[0]
DEMO_LINE, DEMO_DRUG = line_names[DEMO_LINE_ID], drug_names[DEMO_DRUG_ID]

## Sanity check: one held-out pair

Theirs, on the largest validation pair. Means and spreads of the first two PCs, real against generated at guidance 3.

In [ ]:
GUIDANCE_SCALE = 3.0
real = real_cells(val_ds, DEMO_LINE_ID, DEMO_DRUG_ID)
samples = sample(model, scheduler, DEMO_DRUG, DEMO_LINE, len(real), GUIDANCE_SCALE, DATA_DIM, DEVICE, steps=SAMPLE_STEPS, seed=0)
print(f"real validation cells for {DEMO_LINE} + {DEMO_DRUG}: {len(real)}")
print("PC1/PC2 mean -- real:", [round(v, 3) for v in real[:, :2].mean(0).tolist()], "| generated:", [round(v, 3) for v in samples[:, :2].mean(0).cpu().tolist()])
print("PC1/PC2 std  -- real:", [round(v, 3) for v in real[:, :2].std(0).tolist()], "| generated:", [round(v, 3) for v in samples[:, :2].std(0).cpu().tolist()])

## Visual check: real against generated, held-out pairs

Theirs, on the evaluation pairs above.

In [ ]:
import matplotlib.pyplot as plt

OVERLAY_GUIDANCE = 3.0
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for ax, (line_id, drug_id) in zip(axes.flat, EVAL_PAIRS):
    line_name, drug_name = line_names[line_id], drug_names[drug_id]
    real = real_cells(val_ds, line_id, drug_id)
    generated = sample(model, scheduler, drug_name, line_name, len(real), OVERLAY_GUIDANCE, DATA_DIM, DEVICE,
                       steps=SAMPLE_STEPS, seed=0)
    ax.scatter(real[:, 0], real[:, 1], s=10, alpha=0.5, color="tab:blue", label="real (validation)")
    ax.scatter(generated[:, 0].cpu(), generated[:, 1].cpu(), s=10, alpha=0.5, color="tab:orange", label="generated")
    ax.set_title(f"{LINE_TEXT[line_name].split(' ')[0]} + {DRUG_TEXT[drug_name].split(',')[0]} (n={len(real)})", fontsize=9)
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
for ax in axes.flat[len(EVAL_PAIRS):]:
    ax.axis("off")
axes.flat[0].legend(fontsize=8)
fig.suptitle(f"Held-out pairs: real vs. generated at guidance = {OVERLAY_GUIDANCE:g}")
plt.tight_layout(); plt.show()

## Visual check: guidance-scale sweep on one held-out pair

Theirs. At guidance 0 the samples should look like a generic blob unrelated to this pair; as
guidance rises they should pull toward the real cloud; if they collapse to a point at high
guidance, that is the usual guidance-diversity trade-off. The table in the next section puts
numbers on both.

In [ ]:
real_demo = real_cells(val_ds, DEMO_LINE_ID, DEMO_DRUG_ID)
fig, axes = plt.subplots(1, len(GUIDANCE_SWEEP), figsize=(4 * len(GUIDANCE_SWEEP), 4), sharex=True, sharey=True)
for ax, guidance in zip(axes, GUIDANCE_SWEEP):
    generated = sample(model, scheduler, DEMO_DRUG, DEMO_LINE, len(real_demo), guidance, DATA_DIM, DEVICE, steps=SAMPLE_STEPS, seed=0)
    ax.scatter(real_demo[:, 0], real_demo[:, 1], s=10, alpha=0.3, color="tab:gray", label="real (validation)")
    ax.scatter(generated[:, 0].cpu(), generated[:, 1].cpu(), s=10, alpha=0.6, color="tab:red", label="generated")
    ax.set_title(f"guidance = {guidance:g}"); ax.set_xlabel("PC1")
axes[0].set_ylabel("PC2"); axes[0].legend(fontsize=8)
fig.suptitle(f"CFG sweep on a held-out pair: {DEMO_LINE} + {DEMO_DRUG}")
plt.tight_layout(); plt.show()

## Evaluation of the samples

Added. For every evaluation pair and every guidance strength, generate as many cells as the pair
has real validation cells (at most 400) and compare the two populations:

- **MMD**, **energy distance** and the **Fréchet distance on the PCs** (the FID formula without an
  Inception network): how far the generated population is from the real one. Zero means
  indistinguishable.
- **Precision and recall** (k-nearest-neighbour manifolds): the share of generated cells that
  land inside the real cloud, and the share of real cells that the generated cloud covers.
- **Variance ratio**: total variance of the generated cells over that of the real cells. Far
  below 1 means the samples collapsed; guidance tends to push it down.
- **Condition accuracy**: a k-nearest-neighbour vote over real training cells says which drug a
  generated cell looks like; reported next to the same scorer's accuracy on real validation
  cells, which is the ceiling.

Both are reported per pair and averaged, with the sample counts. This is the table the paper's
guidance section reads: the condition metric (distance to the right population) next to the
diversity metric (variance ratio), at several strengths.

In [ ]:
from guidance.eval.cell_metrics import all_metrics, KNNConditionScorer

# Condition scorer: k-nearest-neighbour vote over real training cells labelled with their drug.
SCORER = KNNConditionScorer(train_loader.dataset.pcs, torch.as_tensor(train_loader.dataset.drug_id), k=25, seed=SEED, device=DEVICE)
val_pcs, val_drug = val_ds.pcs, torch.as_tensor(val_ds.drug_id)
SCORER_REAL_ACC = SCORER.accuracy(val_pcs[:20_000], val_drug[:20_000])
print(f"{SCORER.describe()} | drug accuracy on {min(20_000, len(val_pcs)):,} real validation cells: "
      f"{SCORER_REAL_ACC:.1%} (the ceiling; chance {1 / len(drug_names):.1%})")

METRICS = {"eval_pairs": [], "per_guidance": {}, "scorer": SCORER.describe(), "scorer_real_val_drug_accuracy": SCORER_REAL_ACC}
for (l, d) in EVAL_PAIRS:
    METRICS["eval_pairs"].append({"line": line_names[l], "drug": drug_names[d],
                                  "text": f"{DRUG_TEXT[drug_names[d]]}, in {LINE_TEXT[line_names[l]]}",
                                  "n_real": int(len(real_cells(val_ds, l, d)))})
print(f"\n{'guidance':>8} {'MMD':>6} {'energy':>7} {'FD_pc':>7} {'prec':>5} {'rec':>5} {'var':>6} {'acc':>6}   "
      f"(means over {len(EVAL_PAIRS)} held-out pairs; NFE per sample in brackets)")
for w in GUIDANCE_SWEEP:
    rows = []
    for (l, d) in EVAL_PAIRS:
        real = real_cells(val_ds, l, d)
        n = min(len(real), 400)
        gen = sample(model, scheduler, drug_names[d], line_names[l], n, w, DATA_DIM, DEVICE, steps=SAMPLE_STEPS, seed=0)
        m = all_metrics(gen, real, seed=SEED)
        m["cond_acc"] = SCORER.accuracy(gen, torch.full((n,), d))
        rows.append({"line": line_names[l], "drug": drug_names[d], **m})
    g = {"per_pair": rows, "nfe_per_sample": SAMPLE_STEPS if w == 1 else 2 * SAMPLE_STEPS}
    for k in ("mmd", "energy", "frechet_pc", "precision", "recall", "var_ratio", "cond_acc"):
        g[f"{k}_mean"] = float(np.mean([r[k] for r in rows]))
    METRICS["per_guidance"][str(w)] = g
    print(f"{w:>8g} {g['mmd_mean']:>6.3f} {g['energy_mean']:>7.3f} {g['frechet_pc_mean']:>7.2f} {g['precision_mean']:>5.2f} "
          f"{g['recall_mean']:>5.2f} {g['var_ratio_mean']:>6.2f} {g['cond_acc_mean']:>6.1%}   [{g['nfe_per_sample']}]")

## Manifest

In [ ]:
report = {
    "method": "classifier-free guidance on Tahoe cell states; base from Tahoe_PCA_Diffusion_CFG.ipynb",
    "modality": "cells",
    "code_version": GIT_SHA,
    "config": dict(batch_size=BATCH_SIZE, seed=SEED, num_train_timesteps=NUM_TRAIN_TIMESTEPS,
                   cfg_dropout_prob=CFG_DROPOUT_PROB, epochs=EPOCHS, lr=LR, limit=LIMIT,
                   sample_steps=SAMPLE_STEPS, guidance_sweep=GUIDANCE_SWEEP, n_eval_pairs=N_EVAL_PAIRS),
    "conditioning": "drug embedding + cell-line embedding, joint null for CFG",
    "sampler": "ancestral DDPM (diffusers DDPMScheduler, squaredcos_cap_v2), eta = 1",
    "best_val_mse": best_val, "best_epoch": best_epoch, "history": history,
    "params": N_PARAMS, "train_seconds": TRAIN_SECONDS,
    "n_train_cells": len(train_loader.dataset), "n_val_cells": len(validation_loader.dataset),
    "split": "held-out (cell line, drug) pairs, seed 42, ~20%; evaluation pairs are validation pairs",
    "checkpoint_meta": CKPT_META,
    "hardware": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu",
    "device": str(DEVICE),
}
report.update(METRICS)
print(json.dumps({k: v for k, v in report.items() if k not in ("history", "per_guidance")}, indent=2))
with open("manifest_cfg_cells.json", "w") as f:
    json.dump(report, f, indent=2)

## Before reporting any of this

- [ ] Every number is on held-out (cell line, drug) pairs; say so
- [ ] MMD and energy distance reported with the variance ratio at each strength, never alone
- [ ] NFE stated next to each row: `2 x steps` whenever the scale is not 1; the other cell
      notebooks use 50 DDIM steps, so state the sampler difference when the numbers sit together
- [ ] Three sampling seeds on one checkpoint, mean +/- std, stated as such
- [ ] Training time, parameter count and hardware from the manifest